# 실습 4: 센서 하나로 이상 구간 찾기
- 상황: 3라인 계측 기록에서 평소와 다르게 튄 구간을 찾아본다
- 목표: 관리선을 긋고 벗어난 구간을 잡는다

## Step 1. 불러오기와 기본 확인

In [7]:
import pandas as pd

df = pd.read_csv("../../data/04_secom.csv")

print("행 수, 열 수:", df.shape)
print()
print("result 값별 건수:")
print(df["result"].value_counts())
print()
print("result 값별 비율:")
print(df["result"].value_counts(normalize=True))


행 수, 열 수: (1567, 592)

result 값별 건수:
result
양품    1463
불량     104
Name: count, dtype: int64

result 값별 비율:
result
양품    0.933631
불량    0.066369
Name: proportion, dtype: float64


In [8]:
df.head()


,measured_at,sensor_001,sensor_002,sensor_003,sensor_004,sensor_005,sensor_006,sensor_007,sensor_008,sensor_009,...,sensor_582,sensor_583,sensor_584,sensor_585,sensor_586,sensor_587,sensor_588,sensor_589,sensor_590,result
0,19/07/2008 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,양품
1,19/07/2008 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,양품
2,19/07/2008 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,불량
3,19/07/2008 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,양품
4,19/07/2008 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,양품


## Step 2. 관리도라는 말부터

### 용어 풀이 - 관리도 주변

| 말 | 뜻 |
|---|---|
| 관리도 | 값을 시간 순서대로 그려두고, 평소 범위를 벗어나는 순간을 잡는 그림 |
| 관리선 | 평소 범위의 경계선. 위쪽 선을 관리 상한, 아래쪽 선을 관리 하한이라 부른다 |
| 평균 | 값들을 다 더해 개수로 나눈 것. 관리도의 한가운데 선이 된다 |
| 표준편차 | 값들이 평균에서 얼마나 흩어져 있는지. 작으면 몰려 있고 크면 넓게 퍼져 있다 |
| 이상 구간 | 관리선을 벗어난 구간. 벗어났다는 것이지 곧 불량이라는 뜻은 아니다 |
| result (판정) | 검사에서 붙은 양품 또는 불량. 공정이 다 끝난 뒤에 붙는다 |
| SPC | 이런 방식으로 공정을 감시하는 것을 통틀어 부르는 말 |

## Step 3. 센서 하나 고르기

In [9]:
sensor_cols = [c for c in df.columns if c.startswith("sensor_")]

# 빈칸이 없고, 값이 실제로 변하는(고유값이 2개 이상인) 센서 열만 남긴다
no_na = [c for c in sensor_cols if df[c].isna().sum() == 0]
varying = [c for c in no_na if df[c].nunique() > 1]

print("빈칸 없는 센서 열 수:", len(no_na))
print("그 중 값이 변하는 열 수:", len(varying))

pick = varying[:5]

for c in pick:
    mn, mx, nu = df[c].min(), df[c].max(), df[c].nunique()
    print(f"{c} | 범위: {mn} ~ {mx} | 고유값 개수: {nu}")


빈칸 없는 센서 열 수: 52
그 중 값이 변하는 열 수: 52
sensor_021 | 범위: 1.1797 ~ 1.4534 | 고유값 개수: 552
sensor_087 | 범위: 2.2425 ~ 2.5555 | 고유값 개수: 472
sensor_088 | 범위: 0.7749 ~ 0.9935 | 고유값 개수: 249
sensor_089 | 범위: 1627.4714 ~ 2105.1823 | 고유값 개수: 973
sensor_114 | 범위: 0.8534 ~ 0.9763 | 고유값 개수: 468


### 결과 정리 (실행 결과 기준)

| 센서 이름 | 값의 범위 | 고유값 개수 | 판정(종류 수 기준) | 자릿수로 본 성격 |
|---|---|---|---|---|
| sensor_021 | 1.1797 ~ 1.4534 | 552 | 재고 있는 값에 가까움 | 1~2대 소수 — 비율·지수처럼 작은 소수값으로 보임 (정확한 의미는 불확실) |
| sensor_087 | 2.2425 ~ 2.5555 | 472 | 재고 있는 값에 가까움 | 2대 소수 — 위와 비슷한 작은 소수값 (불확실) |
| sensor_088 | 0.7749 ~ 0.9935 | 249 | 재고 있는 값에 가까움 | 0~1 사이 소수 — 비율(%)처럼 보이는 값 (불확실) |
| sensor_089 | 1627.4714 ~ 2105.1823 | 973 | 재고 있는 값에 가까움 | 네 자리 큰 수 — 온도·전압·용량처럼 큰 눈금을 세는 값으로 보임 (불확실) |
| sensor_114 | 0.8534 ~ 0.9763 | 468 | 재고 있는 값에 가까움 | 0~1 사이 소수 — 비율처럼 보이는 값 (불확실) |

참고: 센서 열 590개 중 빈칸이 하나도 없는 열은 52개뿐이었고, 그중 위 5개를 순서대로 뽑았다. 값 종류가 모두 수백 개 이상이라 다섯 열 모두 "정해 넣은 값"보다는 "재고 있는 값"에 가깝다고 판정했다.


[내가 고른 열]<br>
고른 열 : [sensor_089]<br>
값 범위 : [1627.47] ~ [2105.18]   값 종류 : [973]가지<br>
값 종류가 [많으니] 이 열은 [재고 있는 값]에 가깝다<br>
AI가 본 성격 : [세 자리 이상 — 큰 수로 세는 값]

## Step 4. 평소 범위를 정하기
평균에서 표준편차 3배만큼 위아래로 관리선을 긋는다.

In [10]:
# 앞에서 고른 센서 이름을 넣는다 (본인이 고른 것으로 바꾸세요)
센서 = "sensor_089"

# 평균 — 이 센서 값들의 한가운데
평균 = df[센서].mean()

# 표준편차 — 값들이 평균에서 얼마나 흩어져 있는지
표준편차 = df[센서].std()

# 관리 상한 / 하한 — 평균에서 표준편차 3배만큼 위아래
위선 = 평균 + 3 * 표준편차
아래선 = 평균 - 3 * 표준편차

# round(값, 2) — 소수점 둘째 자리까지만 보여준다
print("평균:", round(평균, 2))
print("관리 상한:", round(위선, 2))
print("관리 하한:", round(아래선, 2))

평균: 1807.82
관리 상한: 1968.43
관리 하한: 1647.2


## Step 5. 관리도 그리기
평균·관리 상한·관리 하한을 선으로 긋고, 선을 벗어난 점만 다른 색으로 표시한다. 그림은 `results` 폴더에 저장한다.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# 한글이 깨지지 않게 폰트를 지정한다 (Windows 기본 한글 글꼴)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 이 노트북이 있는 폴더 안에 results 폴더를 만든다 (이미 있으면 그냥 둔다)
os.makedirs("results", exist_ok=True)

값 = df[센서]
순서 = np.arange(len(값))

# 관리선을 벗어난 점만 True
벗어남 = (값 > 위선) | (값 < 아래선)

fig, ax = plt.subplots(figsize=(12, 5))

# 전체 값은 선으로, 벗어난 점만 빨간 점으로 덧그린다
ax.plot(순서, 값, color="steelblue", linewidth=1, label=f"{센서} 값")
ax.scatter(순서[벗어남], 값[벗어남], color="red", zorder=5, label="관리선을 벗어난 점")

ax.axhline(평균, color="green", linestyle="-", label="평균")
ax.axhline(위선, color="darkorange", linestyle="--", label="관리 상한 (UCL)")
ax.axhline(아래선, color="darkorange", linestyle="--", label="관리 하한 (LCL)")

ax.set_title(f"{센서} 관리도")
ax.set_xlabel("측정 순서")
ax.set_ylabel(f"{센서} 값")
ax.legend()

fig.savefig(f"results/{센서}_관리도.png", dpi=150, bbox_inches="tight")
plt.show()

print("관리선을 벗어난 점 개수:", 벗어남.sum())


## Step 6. 벗어난 점의 result 확인
관리선을 벗어난 행만 골라, 그 행들의 result 값을 세고 전체 불량 비율과 비교한다.

In [23]:
# 관리 상한 또는 하한을 벗어난 행만 고른다
벗어난행 = df[(df[센서] > 위선) | (df[센서] < 아래선)]

print("벗어난 행 수:", len(벗어난행))
print()
print("벗어난 행의 result 값별 건수:")
print(벗어난행["result"].value_counts())
print()
print("벗어난 행의 result 값별 비율:")
print(벗어난행["result"].value_counts(normalize=True))
print()
print("전체 result 값별 비율:")
print(df["result"].value_counts(normalize=True))


벗어난 행 수: 7

벗어난 행의 result 값별 건수:
result
양품    6
불량    1
Name: count, dtype: int64

벗어난 행의 result 값별 비율:
result
양품    0.857143
불량    0.142857
Name: proportion, dtype: float64

전체 result 값별 비율:
result
양품    0.933631
불량    0.066369
Name: proportion, dtype: float64


### 결과 정리 (실행 결과 기준)

| 구분 | 양품 비율 | 불량 비율 | 건수 |
|---|---|---|---|
| 관리선을 벗어난 행 | 85.71% | 14.29% | 7건 (양품 6 / 불량 1) |
| 전체 행 | 93.36% | 6.64% | 1,567건 (양품 1,463 / 불량 104) |

벗어난 행에서 불량 비율(14.29%)이 전체 불량 비율(6.64%)보다 약 2배 높다. 다만 벗어난 행이 7건뿐이라 이 차이만으로 sensor_089가 불량과 확실히 관련 있다고 단정하기는 어렵다.

## Step 7. sensor_319 — 관리선 기준 비교 (3배 vs 2배)

In [ ]:
센서 = "sensor_319"

평균 = df[센서].mean()
표준편차 = df[센서].std()

결과 = []
for 배수, 이름 in [(3, "기준 A (3배)"), (2, "기준 B (2배)")]:
    상한 = 평균 + 배수 * 표준편차
    하한 = 평균 - 배수 * 표준편차
    벗어난행 = df[(df[센서] > 상한) | (df[센서] < 하한)]
    불량건수 = (벗어난행["result"] == "불량").sum()
    불량비율 = round(불량건수 / len(벗어난행) * 100, 2) if len(벗어난행) > 0 else None

    결과.append({
        "기준": 이름,
        "관리 상한": round(상한, 4),
        "관리 하한": round(하한, 4),
        "벗어난 행 수": len(벗어난행),
        "그 중 불량 건수": 불량건수,
        "불량 비율(%)": 불량비율,
    })

전체불량비율 = round((df["result"] == "불량").mean() * 100, 2)
print("전체 불량 비율:", 전체불량비율, "%")

결과표 = pd.DataFrame(결과)
결과표


### 결과 정리 (실행 결과 기준)

| 기준 | 관리 상한 | 관리 하한 | 벗어난 행 수 | 그 중 불량 건수 | 불량 비율 | 전체 불량 비율(6.64%)과 비교 |
|---|---|---|---|---|---|---|
| 기준 A (3배) | 6.8366 | -0.459 | 4건 | 0건 | 0.0% | 전체보다 낮음 |
| 기준 B (2배) | 5.6206 | 0.7569 | 45건 | 0건 | 0.0% | 전체보다 낮음 |

sensor_319는 결측치가 1개 있었고(전체 1,567행 중), 벗어난 행으로 잡힌 곳에는 두 기준 모두 불량이 한 건도 없었다. 관리선을 2배로 좁히면 벗어난 행 수는 4건→45건으로 크게 늘지만, 그 안의 불량 비율은 여전히 0%다 — 이번 데이터에서는 sensor_319의 관리선 이탈이 불량과 연결되는 모습을 보이지 않았다.

## Step 8. sensor_090 — 관리선 기준 비교 (3배 vs 2배)

In [25]:
센서 = "sensor_090"

print("결측치 개수:", df[센서].isna().sum())

평균 = df[센서].mean()
표준편차 = df[센서].std()

결과 = []
for 배수, 이름 in [(3, "기준 A (3배)"), (2, "기준 B (2배)")]:
    상한 = 평균 + 배수 * 표준편차
    하한 = 평균 - 배수 * 표준편차
    벗어난행 = df[(df[센서] > 상한) | (df[센서] < 하한)]
    불량건수 = (벗어난행["result"] == "불량").sum()
    불량비율 = round(불량건수 / len(벗어난행) * 100, 2) if len(벗어난행) > 0 else None

    결과.append({
        "기준": 이름,
        "관리 상한": round(상한, 4),
        "관리 하한": round(하한, 4),
        "벗어난 행 수": len(벗어난행),
        "그 중 불량 건수": 불량건수,
        "불량 비율(%)": 불량비율,
    })

전체불량비율 = round((df["result"] == "불량").mean() * 100, 2)
print("전체 불량 비율:", 전체불량비율, "%")

결과표 = pd.DataFrame(결과)
결과표


결측치 개수: 51
전체 불량 비율: 6.64 %


,기준,관리 상한,관리 하한,벗어난 행 수,그 중 불량 건수,불량 비율(%)
0,기준 A (3배),0.3458,0.0316,6,2,33.33
1,기준 B (2배),0.2934,0.0840,7,2,28.57


### 결과 정리 (실행 결과 기준)

| 기준 | 관리 상한 | 관리 하한 | 벗어난 행 수 | 그 중 불량 건수 | 불량 비율 | 전체 불량 비율(6.64%)과 비교 |
|---|---|---|---|---|---|---|
| 기준 A (3배) | 0.3458 | 0.0316 | 6건 | 2건 | 33.33% | 전체보다 약 5배 높음 |
| 기준 B (2배) | 0.2934 | 0.084 | 7건 | 2건 | 28.57% | 전체보다 약 4.3배 높음 |

sensor_090은 결측치가 51개로 앞서 본 다른 센서들보다 많았다(전체 1,567행 중). 두 기준 모두 벗어난 행에서 불량 비율이 전체 불량 비율(6.64%)보다 뚜렷하게 높게 나왔다 — 다만 벗어난 행 수 자체가 6~7건으로 매우 적어서, 이 결과만으로 sensor_090이 불량과 강하게 관련 있다고 단정하기는 어렵다.

## Step 9. sensor_089 — 관리선 기준 비교 (3배 vs 2배)

In [ ]:
센서 = "sensor_089"

print("결측치 개수:", df[센서].isna().sum())

평균 = df[센서].mean()
표준편차 = df[센서].std()

결과 = []
for 배수, 이름 in [(3, "기준 A (3배)"), (2, "기준 B (2배)")]:
    상한 = 평균 + 배수 * 표준편차
    하한 = 평균 - 배수 * 표준편차
    벗어난행 = df[(df[센서] > 상한) | (df[센서] < 하한)]
    불량건수 = (벗어난행["result"] == "불량").sum()
    불량비율 = round(불량건수 / len(벗어난행) * 100, 2) if len(벗어난행) > 0 else None

    결과.append({
        "기준": 이름,
        "관리 상한": round(상한, 4),
        "관리 하한": round(하한, 4),
        "벗어난 행 수": len(벗어난행),
        "그 중 불량 건수": 불량건수,
        "불량 비율(%)": 불량비율,
    })

전체불량비율 = round((df["result"] == "불량").mean() * 100, 2)
print("전체 불량 비율:", 전체불량비율, "%")

결과표 = pd.DataFrame(결과)
결과표


### 결과 정리 (실행 결과 기준)

| 기준 | 관리 상한 | 관리 하한 | 벗어난 행 수 | 그 중 불량 건수 | 불량 비율 | 전체 불량 비율(6.64%)과 비교 |
|---|---|---|---|---|---|---|
| 기준 A (3배) | 1968.4268 | 1647.2032 | 7건 | 1건 | 14.29% | 전체보다 약 2배 높음 |
| 기준 B (2배) | 1914.8895 | 1700.7405 | 81건 | 5건 | 6.17% | 전체와 거의 같음(오히려 살짝 낮음) |

sensor_089는 결측치가 0개였다. 기준을 3배에서 2배로 좁히면 벗어난 행이 7건→81건으로 크게 늘지만, 그만큼 정상 범위의 값들도 많이 섞여 들어와서 불량 비율은 오히려 14.29%→6.17%로 떨어졌다 — 관리선을 너무 좁게 잡으면 "이상 신호"의 의미가 흐려질 수 있음을 보여주는 사례다.

sensor_319, sensor_090, sensor_089 를 차례대로 상한선, 하한선을 각각 3배, 2배로 실행한 결과, 결측치가 1인 sensor_319는 관리선을 2배로 좁히면 벗어난 행수는 4건에서 45건으로 늘지만 그 안의 불량 비율은 여전히 0%, sensor_090 관리선 기준이 2배 였을 때 벗어난 행수가 6건 에서 7건으로 매우 적어서 이 결과만으로 불량과 강하게 관련있다고 단정하기 어려웠고,sensor_089는 관리선을 2배로 좁히면 벗어난 행 수가 7건에서 81건으로 늘지만 불량건수가 5건으로 불량비율이 14.29%에서 6.16%로 떨어졌다. 이런 결과는 sensor_089의 경우 관리선을 너무 좁게 잡으면 "이상신호"의 의미가 흐려질 수 있다고 한다.

day01>lab04_control_chart.ipynb 는 혼자서 집에서 강의안 보고 연습한 파일입니다.